# Tutorial 05: Generating density maps from a synthetic pulsar population

Once a simulated population (or a set of synthetic populations) has been created by running one of the simulator scripts (see Tutorials 01, 02, and 03), we can generate a synthetic representation of this simulation that is readable by a machine-learning pipeline. Depending on the type of simulation that has been performed, three types of generator scripts

* `mlpoppyns/generator/generate_dataset_full.py`
* `mlpoppyns/generator/generate_dataset_surveys.py`
* `mlpoppyns/generator/generate_single_surveys.py`

can be used to produce two-dimensional density maps of our population(s).

We use the first script `mlpoppyns/generator/generate_dataset_full.py` for end-to-end simulations that were obtained
using the `simulate_population_full.py` module. The script will read the corresponding `final_population.pkl.gz` 
output files for each simulated population and generate a density maps in the form of either a `.png` image or 2D 
NumPy array. These maps store the spatial density and velocity information in galactocentric or equatorial 
(ICRS) reference frames for all evolved neutron stars as well as their distribution and corresponding radio fluxes 
in the $P-\dot{P}$ plane.

The second script `mlpoppyns/generator/generate_dataset_surveys.py` is used for simulations that have been run using 
the `simulate_population_magrot_det.py` module. The script will read the corresponding `.pkl.gz` output files that are
produced for each of our simulated surveys for each synthetic population. It then generates a set of density maps in 
the form of either `.png` images or 2D NumPy arrays for those simulated neutron stars that are detected by the modelled 
surveys only. The corresponding density maps store their spatial density and proper motion information in the equatorial 
(ICRS) reference frame and their distribution and corresponding fluxes in the $P-\dot{P}$ plane, respectively.

If we have simulated a single population, we can use the script `mlpoppyns/generator/generate_single_surveys.py`. 
The script will read the corresponding `.pkl.gz` output files that are produced for each of our simulated surveys. 
It then generates a set of density maps in the form of either `.png` images or 2D NumPy arrays for those simulated 
neutron stars that are detected by the modelled surveys only. The corresponding density maps store their spatial 
density and proper motion information in the equatorial (ICRS) reference frame and their distribution and corresponding 
fluxes in the $P-\dot{P}$ plane, respectively.

To run each of these generator scripts, we have to specify the following parameters:

* `data`: the path where the simulated populations are located.
* `save_dir`: the path to the folder where the dataset of maps are saved.
* `data_type`: the type of maps to produce either `array` or `image`. 
    If `array`, the generator will produce the density maps in the form of 2D `.npy` arrays; 
    if `image`, it will produce them in the form of `.png` images.
* `resolution_dyn`: the resolution in bins or pixels of the maps containing the dynamical information.
* `resolution_ppdot`: the resolution in bins or pixels of the maps containing the $P-\dot{P}$ information.

Suppose that we have created a dataset of simulated populations that is stored in `data/example_simulation_helper_magrot` 
using the `simulate_population_magrot_det.py` script. Let us assume that we want to create a map dataset of 2D arrays 
for the spatial and velocity information with a resolution of $32 \times 16$ and the density and radio flux maps in 
the $P-\dot{P}$ plane with a resolution of $32 \times 32$. To obtain these maps, we run the following command:
```commandline
python mlpoppyns/generator/generate_dataset_surveys.py --data data/example_simulation_helper_magrot --save_dir output/generator --data_type array --resolution_dyn 32 --resolution_ppdot 32
```
By running the script, we create a folder `output/generator` where a set of 2D arrays are 
stored for each simulated population (sample). In addition, this produces a single `dataset_full.csv` and 
`statistics_full.json` file containing information about the dataset and statistical information for each label,
respectively. 

The CSV file provides one line for each synthetic simulation sample in the dataset indicating the file path for its 2D arrays (these are potential input channels for the machine learning pipeline) and the numerical values used to generate the underlying population (these will serve as labels for the machine learning). 

The JSON file contains information about the mean, standard deviation, minimum and maximum values for each of the dataset labels. We use this statistical information during the training process if one wants to normalize or standardize the label values.

In [ ]:
import argparse
import collections
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg
from mlpoppyns.generator.generate_dataset_surveys import generate_dataset
from mlpoppyns.generator.dataset_splitter import main

WARNING: If you see a warning here, make sure to set the path to the repository in the simulator configuration file. Otherwise, the examples below will not run.

## Setting up and running the generator

Before running the generator script, we configure it by passing several arguments. This includes the type of representation for the maps (either `array` or `image`) and the resolutions for the dynamical and $P-\dot{P}$ maps.

We also specify the location of the synthetic simulations we want to convert into density maps and the output directory where the simulation results will be saved.

In [ ]:
dataset_dir = "../../data/example_simulation_helper_magrot"
output_dir = "output/generator"

We run the generator by calling the `generate_dataset` function from the `mlpoppyns.generator.generate_dataset_surveys` module for a specific choice of parameters.

In [ ]:
generator_args = argparse.Namespace(
    data=dataset_dir,
    save_dir=output_dir,
    data_type="array",
    resolution_dyn=32,
    resolution_ppdot=32,
)

In [ ]:
generate_dataset(generator_args)

## Visualizing the generated maps

To display the maps, we first load the corresponding `.npy` arrays.

In [ ]:
ppdot_PMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_ppdot_map_0.npy")
)
ppdot_HTRU = np.load(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_ppdot_map_0.npy")
)
ppdot_SMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_ppdot_map_0.npy")
)
ppdot_flux_PMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_ppdot_map_fluxes_0.npy")
)
ppdot_flux_HTRU = np.load(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_ppdot_map_fluxes_0.npy")
)
ppdot_flux_SMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_ppdot_map_fluxes_0.npy")
)
position_PMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_position_map_radec_0.npy")
)
position_HTRU = np.load(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_position_map_radec_0.npy")
)
position_SMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_position_map_radec_0.npy")
)

Transposing the maps for correct visualization with imshow.

In [ ]:
ppdot_PMPS = ppdot_PMPS.T
ppdot_HTRU = ppdot_HTRU.T
ppdot_SMPS = ppdot_SMPS.T
ppdot_flux_PMPS = ppdot_flux_PMPS.T
ppdot_flux_HTRU = ppdot_flux_HTRU.T
ppdot_flux_SMPS = ppdot_flux_SMPS.T
position_PMPS = position_PMPS.T
position_SMPS = position_SMPS.T
position_HTRU = position_HTRU.T

Producing the $P-\dot{P}$ maps for each of the three surveys.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_PMPS, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"$P-\dot{P}$ density map: PMPS", fontsize=30, x=0.6)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_SMPS, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"$P-\dot{P}$ density map: SMPS", fontsize=30, x=0.6)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_HTRU, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"$P-\dot{P}$ density map: HTRU", fontsize=30, x=0.6)

plt.show()

Producing the $P-\dot{P}$ maps weighted with the radio fluxes for each of the three surveys.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_flux_PMPS, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label(r"Average $\log_{10}(S_{\rm radio} \, [{\rm Jy}])$")
fig.suptitle(
    r"Fluxes displayed in $P-\dot{P}$ space: PMPS", fontsize=30, x=0.6
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_flux_SMPS, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label(r"Average $\log_{10}(S_{\rm radio} \, [{\rm Jy}])$")
fig.suptitle(
    r"Fluxes displayed in $P-\dot{P}$ space: SMPS", fontsize=30, x=0.6
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_flux_HTRU, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label(r"Average $\log_{10}(S_{\rm radio} \, [{\rm Jy}])$")
fig.suptitle(
    r"Fluxes displayed in $P-\dot{P}$ space: HTRU", fontsize=30, x=0.6
)

plt.show()

Generating the sky position maps.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_PMPS, cmap="viridis", origin="lower")
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"Sky density map: PMPS", fontsize=30, x=0.5)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_SMPS, cmap="viridis", origin="lower")
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"Sky density map: SMPS", fontsize=30, x=0.5)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_HTRU, cmap="viridis", origin="lower")
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"Sky density map: HTRU", fontsize=30, x=0.5)

plt.show()

## Splitting the dataset for training, validation and testing

We can also split the dataset into training/validation, training/test or into training/validation/test sets. To do this,
we use the `dataset_splitter.py` script in the `mlpoppyns/generator` folder.

As we are using a small dataset in this example, we will only split the dataset into training and testing subsets for simplicity. To this end, we specify a fraction of the total dataset that will form the test subset by passing the argument `test_split` in the 
`dataset_splitter.py` script. For example:
```commandline
python mlpoppyns/generator/dataset_splitter.py --dataset_path output/generator --test_split 0.2
```
This will create two files `dataset_train.csv` and `dataset_valid.csv` in the location of the `dataset_full.csv` file
that will specify the specific simulation samples belonging to the train dataset (80% of the total dataset in this case)
and those belonging to the validation dataset (20% of the total dataset). This split is performed by randomly sampling 
the validation subset from the total dataset according to the specified split. We also produce a file
`statistics_train.json` (saved in the same location) that contains the statistics computed on the training labels only.

Setting up and running the splitter.

In [ ]:
splitter_args = argparse.Namespace(
    dataset_path=output_dir, valid_split=None, test_split=0.2
)
main(splitter_args)